# Strike a Pose: Kaggle run

Quickstart section 4 (FR-029, SC-010). Run this notebook on a GPU session with the private datasets `sap-src` (the bundle from `scripts/build_kaggle_bundle.sh`) and `sap-assets` attached.

- The first session (no previous output attached) runs the extrapolation gate. The notebook stops when the gate fails.
- A later session attaches the previous version's output. The notebook copies `out/` into `/kaggle/working/out` and resumes.
- Each session stops at a checkpoint before 8.5 hours (`--time-budget 8.5h`).

In [ ]:
import json
import os
import shutil
import subprocess
import sys

# Set before any torch or sap process starts, so every child process inherits them.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["SAP_ASSET_ROOT"] = "/kaggle/input/sap-assets"
os.environ["SAP_OUTPUT_DIR"] = "/kaggle/working/out"
os.environ["SAP_BUNDLE"] = "/kaggle/input/sap-src"
os.environ["SAP_CONSTRAINTS"] = "/kaggle/input/sap-src/constraints.txt"

print("python:", sys.version.split()[0])

In [ ]:
# Install the wheel from the bundle, pinned by constraints.txt (research R11).
# Keep the Kaggle image's own CUDA build of torch: drop the torch pin before installing.
!grep -v '^torch==' "$SAP_CONSTRAINTS" > /kaggle/working/pins.txt
!pip install --quiet -c /kaggle/working/pins.txt /kaggle/input/sap-src/strike_a_pose-*.whl

In [ ]:
# Print the versions and the GPU, then check the installed versions against the bundled constraints.
import torch

print("python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
!sap info --constraints "$SAP_CONSTRAINTS" --strict

In [ ]:
from pathlib import Path

BUNDLE = Path(os.environ["SAP_BUNDLE"])
WORKING = Path("/kaggle/working")
NOT_PREVIOUS_OUTPUT = {"sap-src", "sap-assets"}

# A previous output is a notebook-output input: a folder under /kaggle/input that holds out/.
previous_outputs = [
    path / "out"
    for path in Path("/kaggle/input").iterdir()
    if path.is_dir() and path.name not in NOT_PREVIOUS_OUTPUT and (path / "out").is_dir()
]
print("previous output attached:", [str(p) for p in previous_outputs] or "no")

In [ ]:
# Gate cell (quickstart section 4 step 0). It runs only when no previous output is attached.
# Stop the notebook when the predicted full prediction time is 12 hours or more.
GATE_LIMIT_SECONDS = 12 * 3600
# Full mesh evaluations / tiny mesh evaluations: 1,440,000 / 9,216 (quickstart section 4 step 0).
MESH_EVALUATION_RATIO = 1_440_000 / 9_216
# SMPL-X faces / stand-in faces: 20,908 SMPL-X faces (public SMPL-X model), 8,448 (R4).
FACE_RATIO = 20_908 / 8_448

if previous_outputs:
    print("gate skipped: a previous output is attached")
else:
    os.chdir(BUNDLE)
    tiny = subprocess.run(
        [
            "sap", "run", "--config", "configs/tiny.yaml",
            "--set", "device=auto", "--out", "/kaggle/working/tiny",
        ],
        check=False,
    )
    if tiny.returncode != 0:
        raise RuntimeError(
            f"gate: the tiny run failed with exit {tiny.returncode}; the Kaggle run does not start"
        )

    os.chdir(WORKING)
    record = json.loads((WORKING / "tiny" / "run_record.json").read_text(encoding="utf-8"))
    if record["hardware_class"] != "gpu":
        raise RuntimeError(
            f"gate: the tiny run used hardware class {record['hardware_class']!r}, not gpu"
        )
    tiny_predict = record["timings"]["predict"]
    if tiny_predict is None:
        raise RuntimeError("gate: the tiny run recorded no timings.predict")

    predicted = tiny_predict * MESH_EVALUATION_RATIO * FACE_RATIO
    torch_version = record["versions"].get("torch")
    print(f"gate: tiny timings.predict = {tiny_predict:.1f} s on torch {torch_version}")
    print(f"gate: predicted full prediction time = {predicted / 3600:.2f} h (limit 12 h)")
    if predicted >= GATE_LIMIT_SECONDS:
        raise RuntimeError(
            f"gate FAILED: predicted {predicted / 3600:.2f} h is 12 h or more. "
            "Apply a research R13 mitigation (predict.n_samples, measure.step_cm, or "
            "predict.measure_mode), record it in the run record, and rerun."
        )
    print("gate PASSED: the full run may start")

In [ ]:
# Resume copy: when a previous output is attached, copy its out/ into the working directory.
if previous_outputs:
    source = previous_outputs[0]
    shutil.copytree(source, WORKING / "out", dirs_exist_ok=True)
    print("copied", source, "to /kaggle/working/out")
else:
    print("no previous output: the full run starts fresh")

In [ ]:
# Full run. --resume continues from the last shard, epoch, or cell.
# --time-budget stops the run at a checkpoint before the session cap.
os.chdir(BUNDLE)
full = subprocess.run(
    [
        "sap", "run", "--config", "configs/full.yaml", "--out", "/kaggle/working/out",
        "--resume", "--time-budget", "8.5h",
    ],
    check=False,
)
os.chdir(WORKING)
print("sap run exit status:", full.returncode)

In [ ]:
# Print the verdict. The run is complete when verdict/verdict.md and report/results.md both exist.
verdict = WORKING / "out" / "verdict" / "verdict.md"
report = WORKING / "out" / "report" / "results.md"
if verdict.exists():
    print(verdict.read_text(encoding="utf-8").splitlines()[0])
else:
    print("verdict not written yet: attach this output as an input and run again")
print("report/results.md present:", report.exists())